# COE Scenario Generation
Generate creative scenarios for error chains using GPT-4o batch API.


In [40]:
import os
import sys
import json
from pathlib import Path
from tokens import azure_key

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm.metrics.utils.e_gen_scenario import COEScenarioGenerator


In [41]:
# Initialize generator
# MODEL_NAME = "Qwen3-VL-8B-Instruct"
# MODEL_NAME = "Qwen3-VL-4B-Instruct"
MODEL_NAME = "llava-1.5-7b-hf"
# MODEL_NAME = "instructblip-vicuna-7b"


DATASET = "aokvqa" 
# DATASET = "fvqa"  

gen = COEScenarioGenerator(
    dataset_name=DATASET,
    model_name=MODEL_NAME,
    azure_key=azure_key,
    merge_chains=True,
)


COEScenarioGenerator: llava-1.5-7b-hf/aokvqa, merge=True, 20 batches


## Step 1: Load COE results and prepare batch input


In [42]:
# Load COE prediction results
coe_path = f"results/pred_postedit/baseline/{MODEL_NAME}/{DATASET}/coe_prediction.json"
with open(coe_path, "r") as f:
    coe_results = json.load(f)

print(f"Loaded {len(coe_results)} COE results from {coe_path}")

# Check how many have error chains
with_errors = [r for r in coe_results if gen._get_error_chains(r)]
print(f"Samples with error chains: {len(with_errors)}/{len(coe_results)}")

# Preview a few
for r in with_errors[:3]:
    chains = gen._get_error_chains(r)
    print(f"uid={r['uid']}: {len(chains)} error chains")

# Preview the prompt for one sample
sample = with_errors[1]
chains = gen._get_error_chains(sample)
print(f"Sample has {len(chains)} error chains. Showing first:\n")
print("=== PROMPT ===")
print(gen.format_prompt(chains[0]["chain"]))

Loaded 7197 COE results from results/pred_postedit/baseline/llava-1.5-7b-hf/aokvqa/coe_prediction.json
Samples with error chains: 5219/7197
uid=1: 1 error chains
uid=6: 1 error chains
uid=9: 1 error chains
Sample has 1 error chains. Showing first:

=== PROMPT ===
Given these visual facts:
"The image shows a sign advertising a show. The sign mentions a Broadway show. Broadway shows are typically performed in theaters. Where would one most likely see the show advertised in the poster is theater."

Generate 3 different creative scenarios where ALL these facts would be visually true.

Requirements:
- Each scenario must be exactly one sentence (less than 20 words)
- Be creative but plausible
- Describe what would be visible in the image
- Do not contradict the given facts

Examples:
Visual facts: "A person is standing on a board. There are waves around."
1. A surfer rides a wave at a tropical beach during sunset.
2. A wakeboarder is pulled behind a speedboat on a calm lake.
3. A paddleboard

In [43]:
# Generate batch request files (uncomment to run)
gen.run_input(coe_results, max_sentences=None)  # or max_sentences=5 to filter


Created 5219 requests in data/coe_gen_merge/llava-1.5-7b-hf/aokvqa/requests/batch.jsonl


## Step 2: Test with ONE batch before firing all


In [44]:
# Check batch files created
batch_files = list(gen.batch_dir.glob("batch_*.jsonl"))
print(f"Batch files: {len(batch_files)}")
for bf in sorted(batch_files)[:5]:
    with open(bf) as f:
        n_lines = len(f.readlines())
    print(f"  {bf.name}: {n_lines} requests")
    
# Preview first request in batch_0
batch_0 = gen.batch_dir / "batch_0.jsonl"
if batch_0.exists():
    with open(batch_0) as f:
        first_req = json.loads(f.readline())
    print(json.dumps(first_req, indent=2))  # truncate if too long
# Submit ONLY batch 0 first (uncomment to run)
gen._run_request_batch(0)



Batch files: 20
  batch_0.jsonl: 261 requests
  batch_1.jsonl: 261 requests
  batch_10.jsonl: 261 requests
  batch_11.jsonl: 261 requests
  batch_12.jsonl: 261 requests
{
  "custom_id": "1_[0,1,2,3]",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4o-batch",
    "messages": [
      {
        "role": "system",
        "content": "You generate creative visual scenarios from given facts."
      },
      {
        "role": "user",
        "content": "Given these visual facts:\n\"The image shows a man standing by some bags on the street. A train would not be on the street. The man would not have luggage waiting for a delivery on the street. The skateboarder is present and not paying attention to the man. What is the man by the bags awaiting is cab.\"\n\nGenerate 3 different creative scenarios where ALL these facts would be visually true.\n\nRequirements:\n- Each scenario must be exactly one sentence (less than 20 words)\n- Be creative but plausible\n- Desc

In [45]:
# Get batch 0 results (after it completes)
try:
    results_0 = gen._get_response_batch(0)
    print(f"Got {len(results_0)} results from batch 0")
    
    # Preview first few
    for r in results_0[:3]:
        print(f"\nuid={r['uid']}:")
        for i, s in enumerate(r['scenarios'], 1):
            print(f"  {i}. {s}")
except Exception as e:
    print(f"Error: {e}")


Got 261 results from batch 0

uid=245:
  1. Children stomp through the muddy ground of a forest trail after a heavy rainstorm.
  2. Hikers navigate a swampy path surrounded by dense greenery and dripping trees.
  3. Farmers wade through mud-filled fields while planting crops during a cloudy, misty afternoon.

uid=203:
  1. Cross-shaped grass wreaths hang from a wooden chapel door, swaying gently in the evening breeze.
  2. Grass wreath crosses are displayed on a rustic wall during a countryside harvest festival celebration.
  3. Two cross-shaped grass wreaths are suspended on a string under lanterns at a solemn outdoor memorial event.

uid=17:
  1. A designer in a blue blouse works on a sleek Macintosh computer in a minimalist studio filled with sketches.
  2. A woman dressed in blue browses a Macintosh computer at a bustling tech store with modern display tables.
  3. A programmer wearing blue codes on a Macintosh computer in a cozy apartment with plants and warm lighting.


## Step 3: Fire all batches (after testing batch 0)


In [46]:
# Submit all remaining batches (uncomment to run)
gen.run_request()

In [47]:
# Check status of all submitted batches
for b in range(gen.n_batches):
    meta_path = gen.meta_dir / f"meta_{b}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        try:
            job = gen.client.batches.retrieve(meta["job_id"])
            print(f"Batch {b}: {job.status}")
        except Exception as e:
            print(f"Batch {b}: error - {e}")


Batch 0: completed
Batch 1: completed
Batch 2: completed
Batch 3: completed
Batch 4: completed
Batch 5: completed
Batch 6: completed
Batch 7: completed
Batch 8: completed
Batch 9: completed
Batch 10: completed
Batch 11: completed
Batch 12: completed
Batch 13: completed
Batch 14: completed
Batch 15: completed
Batch 16: completed
Batch 17: completed
Batch 18: completed
Batch 19: completed


In [48]:
# Resubmit failed batches if needed (uncomment and modify list)
# gen.resubmit_request([0, 1, 2])  # list of batch indices to resubmit


## Step 4: Get all results


In [49]:
# Count NULL responses across ALL batches
total_null = 0
total_responses = 0

for b in range(gen.n_batches):
    try:
        meta = gen._load_meta(b)
        job = gen.client.batches.retrieve(meta["job_id"])
        content = gen.client.files.content(job.output_file_id)
        
        batch_null = 0
        batch_total = 0
        for line in content.iter_lines():
            if not line:
                continue
            batch_total += 1
            payload = json.loads(line)
            text = payload.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content")
            if text is None:
                batch_null += 1
        
        total_null += batch_null
        total_responses += batch_total
        if batch_null > 0:
            print(f"Batch {b}: {batch_null}/{batch_total} filtered")
    except Exception as e:
        print(f"Batch {b}: error - {e}")

print(f"\n=== TOTAL: {total_null}/{total_responses} filtered ({100*total_null/total_responses:.2f}%) ===")

Batch 0: 1/261 filtered
Batch 4: 1/261 filtered
Batch 9: 1/261 filtered
Batch 11: 2/261 filtered
Batch 16: 1/261 filtered
Batch 18: 1/261 filtered

=== TOTAL: 7/5219 filtered (0.13%) ===


In [50]:
# Get all scenarios (after all batches complete)
all_scenarios = gen.get_scenarios()
print(f"Total scenarios: {len(all_scenarios)}")

# Preview results
for r in all_scenarios[:3]:
    print(f"\nuid={r['uid']}, indices={r['indices']}:")
    for i, s in enumerate(r['scenarios'], 1):
        print(f"  {i}. {s}")


Total scenarios: 5219

uid=245, indices=[0, 1, 2]:
  1. Children stomp through the muddy ground of a forest trail after a heavy rainstorm.
  2. Hikers navigate a swampy path surrounded by dense greenery and dripping trees.
  3. Farmers wade through mud-filled fields while planting crops during a cloudy, misty afternoon.

uid=203, indices=[0, 1, 2]:
  1. Cross-shaped grass wreaths hang from a wooden chapel door, swaying gently in the evening breeze.
  2. Grass wreath crosses are displayed on a rustic wall during a countryside harvest festival celebration.
  3. Two cross-shaped grass wreaths are suspended on a string under lanterns at a solemn outdoor memorial event.

uid=17, indices=[0, 1, 2]:
  1. A designer in a blue blouse works on a sleek Macintosh computer in a minimalist studio filled with sketches.
  2. A woman dressed in blue browses a Macintosh computer at a bustling tech store with modern display tables.
  3. A programmer wearing blue codes on a Macintosh computer in a cozy ap

In [51]:
df = gen.save_parquet()
print(len(df))
len(df.uid.unique())

Saved 5219 rows to data/coe_gen_merge/parquet/llava-1.5-7b-hf_aokvqa.parquet
5219


5219

In [54]:
df

,uid,indices,scenario_1,scenario_2,scenario_3
0,245,"0,1,2",Children stomp through the muddy ground of a f...,Hikers navigate a swampy path surrounded by de...,Farmers wade through mud-filled fields while p...
1,203,"0,1,2",Cross-shaped grass wreaths hang from a wooden ...,Grass wreath crosses are displayed on a rustic...,Two cross-shaped grass wreaths are suspended o...
2,17,"0,1,2",A designer in a blue blouse works on a sleek M...,A woman dressed in blue browses a Macintosh co...,A programmer wearing blue codes on a Macintosh...
3,138,"0,2,3",A yellow and white kite shaped like a squid gl...,Children cheer as a kite resembling a giant sq...,A yellow and white squid-like kite floats abov...
4,139,"0,1,2",A conference room features a left chair with t...,A trade expo booth showcases a left chair with...,A boardroom setting includes a left chair with...
...,...,...,...,...,...
5214,18171,3,A sparsely occupied stadium hosts an early-sea...,"At a concert venue, a singer rehearses onstage...",A school awards ceremony ends as students and ...
5215,18151,1,A fuzzy-coated Hereford cow stands in a sunlit...,"A brown Hereford cow rests near a wooden barn,...",A group of fuzzy-coated Hereford cows huddle u...
5216,18116,"0,1,2,3",A skateboarder in black executes a dramatic ha...,A breakdancer in black performs a gravity-defy...,A parkour athlete in black maneuvers into a ha...
5217,18179,0,A lavish Victorian parlor features gold chairs...,A grand Victorian ballroom is set for a gala w...,An antique shop displays Victorian gold chairs...


In [ ]:
# from huggingface_hub import HfApi, create_repo, upload_folder
# from huggingface_hub.utils import disable_progress_bars
# disable_progress_bars()
# from tokens import HF_TOKEN

# api = HfApi(token=HF_TOKEN)
# repo_id = "JJoy333/RationaleVQA"
# create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

# merge_folder = "lite" if gen.merge_chains else "full"
# upload_folder(
#     folder_path=str(gen.parquet_dir),
#     repo_id=repo_id,
#     repo_type="dataset",
#     path_in_repo=f"coe_gen/{merge_folder}"
# )


CommitInfo(commit_url='https://huggingface.co/datasets/JJoy333/RationaleVQA/commit/7bb34e22a6091911434fe56e1191dd98ef63488a', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7bb34e22a6091911434fe56e1191dd98ef63488a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JJoy333/RationaleVQA', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JJoy333/RationaleVQA'), pr_revision=None, pr_num=None)